# R01. Output Weights
- This predicts the impact of plate appearance outputs on a player's fantasy points, useful for weighting outputs for M03. Plate Appearance model evaluations
- Type: Research
- Run Frequency: Irregular
- Sources:
    - A10. Player Results
- Created: 3/2/2025
- Updated: 3/2/2025

Since plate appearance models predict a variety of possible outputs, choosing a single metric on which to evaluate accuracy is impossible. <br>
However, since the ultimate goal is to accurately predict fantasy points, FP is a decent substitute. <br>
One could weigh singles as worth 3 FP, HR as 10, etc..., but this ignores the impact of RBI, R, and SB. <br>
The regressions below allow for proper weighting based on the FP a player is likely to score throughout the game as a result of that PA result.

### Imports

In [ ]:
%run "U01. Imports.ipynb"
%run "U02. Functions.ipynb"
%run "U03. Classes.ipynb"
%run "U04. Datasets.ipynb"
%run "U05. Models.ipynb"

### Batters

##### Data

In [ ]:
%%time
# List to hold all dataframes
batters_data = []

# Loop through all files in the 'A10. Player Results' directory and its subdirectories
for root, dirs, files in os.walk(os.path.join(baseball_path, 'A10. Player Results')):
    for filename in files:
        if 'batters' in filename and filename.endswith('.csv'):
            file_path = os.path.join(root, filename)
            df = pd.read_csv(file_path)
            batters_data.append(df)

# Concatenate all dataframes into one
batters_combined = pd.concat(batters_data, ignore_index=True)

##### Create Variables

In [ ]:
batters_combined['singles'] = batters_combined['h'] - batters_combined[['doubles', 'triples', 'hr']].sum(axis=1)

In [ ]:
batters_combined['runs_produced'] = batters_combined['rbi'] + batters_combined['r'] - batters_combined['hr']
batters_combined['runs_produced2'] = (batters_combined['rbi'] + batters_combined['r']) / 2

##### Regression

In [ ]:
X = batters_combined[['singles', 'doubles', 'triples', 'hr', 'bb', 'hbp']]
y = batters_combined['fp']

model = sm.OLS(y, X).fit()

print(model.summary())

### Pitchers

##### Data

In [ ]:
%%time
# List to hold all dataframes
pitchers_data = []

# Loop through all files in the 'A10. Player Results' directory and its subdirectories
for root, dirs, files in os.walk(os.path.join(baseball_path, 'A10. Player Results')):
    for filename in files:
        if 'pitchers' in filename and filename.endswith('.csv'):
            file_path = os.path.join(root, filename)
            try:
                df = pd.read_csv(file_path)
            except:
                pass
            pitchers_data.append(df)

# Concatenate all dataframes into one
pitchers_combined = pd.concat(pitchers_data, ignore_index=True)

In [ ]:
pitchers_combined.head()

##### Create Variables

Note: we can't use this data to distinguish between non-strikeout outs or non-hr hits

In [ ]:
pitchers_combined['non_strikeout'] = pitchers_combined['outs'] - pitchers_combined['k']
pitchers_combined['non_hr'] = pitchers_combined['h'] - pitchers_combined['hr']

##### Regression

In [ ]:
X = pitchers_combined[['k', 'non_strikeout', 'non_hr', 'hr', 'bb']]
y = pitchers_combined['fp']

model = sm.OLS(y, X).fit()

print(model.summary())